In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Document loader
from langchain_community.document_loaders import PyPDFDirectoryLoader

# HuggingFace Embeddings (FREE)
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline






c:\Users\asus\Desktop\medical_chatbot\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def read_docs(directory):
    loader = PyPDFDirectoryLoader(directory)
    documents = loader.load()
    return documents

In [6]:
def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    return chunks

In [7]:
docs = read_docs("documents/")
print("Total documents:", len(docs))

Total documents: 4505


In [8]:
documents = chunk_data(docs)
print("Total chunks:", len(documents))

Total chunks: 25859


In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embedding
vector = embeddings.embed_query("How are you?")
print("Embedding vector length:", len(vector))


C:\Users\asus\AppData\Local\Temp\ipykernel_14612\1253428448.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embedding vector length: 384


In [10]:
from langchain_community.vectorstores import Pinecone
from pinecone import Pinecone as PineconeClient


from pinecone import Pinecone as PineconeClient
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings



In [11]:
import os
from pinecone import Pinecone
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone()

pinecone_index = pc.Index(
    name="medicalbot",
    host="https://medicalbot-8dmpk8w.svc.aped-4627-b74a.pinecone.io"
)



In [12]:
vectorstore = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    index_name="medicalbot" 
)

print("✅ Documents successfully stored in Pinecone")


✅ Documents successfully stored in Pinecone


In [13]:
# ============================================
# 6. RETRIEVER
# ============================================

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def get_context_from_pinecone(query):
    docs = retriever.invoke(query)   
    context = "\n".join([doc.page_content for doc in docs])
    return context



In [14]:
# ============================================
# 7. HUGGINGFACE LLM (ANSWER GENERATION)
# ============================================

qa_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=512
)


Device set to use cpu


In [15]:
def generate_answer(query):
    context = get_context_from_pinecone(query)

    prompt = f"""
Answer the medical question clearly and completely.
Use only the information from the context.
Do not repeat the question or the context.

Context:
{context}

Question:
{query}
"""

    raw_output = qa_pipeline(prompt)[0]["generated_text"]

    # Clean output
    answer =raw_output.split("Question:")[-1].strip()

    return answer


In [18]:
print(generate_answer("What are ophthalmic antibiotics?"))

medicines that kill bac- teria that cause eye infections.


In [ ]:
# ============================================
# 8. CHATBOT LOOP
# ============================================

while True:
    question = input("\n🧑 Ask a medical question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    answer = generate_answer(question)

    print("\n🩺 Medical Chatbot Answer:\n")
    print(answer)


🩺 Medical Chatbot Answer:

medicines that kill bac- teria that cause eye infections.

🩺 Medical Chatbot Answer:

an autosomal dominant mode of inheritance and genetic heterogeneity.’’ Biotech WeekJuly 28, 2004: 65. Naiglin. L., et al. ‘‘Familial high myopia: evidence of an autosomal dominant mode of inheritance and genetic heterogeneity.’’ Annals of Genetics42, no. 3 (1999): 140-146. Pacell, R, et al. ‘‘Role of genetic factors in the etiology of juvenile-onset myopia based on a longitudinal study of refractive error.’’ Optometry and Visual Science76 (June 1999): 381-386. Saw, SM, et al. ‘‘Myopia: gene-environment interaction.’’ Annals of the Academy of Medicine of Singapore29 (May 2000): 290-297. ‘‘Scientists close to identifying myopia gene.’’ Chemistry and Industry August 16, 2004: 7. Wu, MM, and MH Edwards. ‘‘The effect of having myopic parents: an analysis of myopia in three generations.’’

🩺 Medical Chatbot Answer:

tickle in the throat, runny nose, and sneezing

🩺 Medical Chatbo